###### Notebook 1 — 03_Load_Silver_to_Gold_Augment.py

In [0]:
# Load shared pipeline config (workspace file — not a notebook, so exec instead of %run)
exec(open("/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline/config/pipeline_config.py").read())

In [0]:
# # Load metrics_query (workspace file)
# exec(open("/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline/utils/metrics_query.py").read())

In [0]:
# mq = MetricsQuery(spark)
# print(":white_check_mark: MetricsQuery loaded")

In [0]:
%run /Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline/utils/cdc_validator

In [0]:
# Load audit logger (workspace file — not a notebook, so exec instead of %run)
exec(open("/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline/utils/audit_logger.py").read())

In [0]:
# OPT: Diagnostic CDF scan disabled for production — was adding ~15-30s
# scanning DESCRIBE HISTORY + reading full CDF for events (200K+ records)
# just to print counts. Set DEBUG_CDF_SCAN = True to re-enable.

# DEBUG_CDF_SCAN = False

# if DEBUG_CDF_SCAN:
#     from pyspark.sql.functions import col

#     def get_latest_change_version_and_cdc_count(catalog_name, schema_name, table_name):
#         """
#         Finds the latest version of a Delta table where a merge, update, insert, or delete operation occurred,
#         and returns the count of CDC records since that version.
#         """
#         full_table_name = f"{catalog_name}.{schema_name}.{table_name}"
#         try:
#             history_df = spark.sql(f"DESCRIBE HISTORY {full_table_name}")
#             relevant_ops = ["MERGE", "UPDATE", "INSERT", "DELETE"]
#             filtered_df = history_df.filter(col("operation").isin(relevant_ops))
#             latest_version_row = filtered_df.orderBy(col("version").desc()).limit(1).collect()
#             if latest_version_row:
#                 latest_version = latest_version_row[0]["version"]
#                 tbl_props = spark.sql(f"SHOW TBLPROPERTIES {full_table_name}")
#                 cdc_enabled = tbl_props.filter(col("key") == "delta.enableChangeDataFeed").collect()
#                 if cdc_enabled and cdc_enabled[0]["value"].lower() == "true":
#                     cdc_df = spark.read.format("delta").option("readChangeData", "true") \
#                         .option("startingVersion", latest_version).table(full_table_name)
#                     #cdc_count = cdc_df.count()
#                     #print(f"{full_table_name} CDC ENABLED , Changed records : {cdc_count}")
#                 else:
#                     #print(f"{full_table_name} CDC DISABLED , Changed records : N/A")
#             else:
#                 print(f"{full_table_name} — No MERGE/UPDATE/INSERT/DELETE operations found in history.")
#         except Exception as e:
#             print(f"Error reading CDC for {full_table_name}: {e}")

#     get_latest_change_version_and_cdc_count('hgi','silver','accounts')
#     get_latest_change_version_and_cdc_count('hgi','silver','contacts')
#     get_latest_change_version_and_cdc_count('hgi','silver','events')
# else:
#     print("CDF diagnostic scan skipped (DEBUG_CDF_SCAN = False)")

In [0]:
# ===================================================================================
# OPTIMIZED GOLD AUGMENT PIPELINE — Merged cells 8+9+10
# Key optimizations:
#   1. Guard CREATE TABLE with tableExists() — avoids 3 DDL roundtrips
#   2. Replace .count() with len(df.head(1)) — avoids full action
#   3. Batch all ALTER TABLE TBLPROPERTIES into one call
#   4. Remove duplicate version checks — reuse CDF pre-filter values
#   5. Fix duration_ms not defined error
#   6. No .cache()/.persist() (not supported on serverless)
# ===================================================================================

from pyspark.sql.functions import col, lit, lower, when, current_timestamp, coalesce, split, broadcast
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import time

# ===================================================================================
# HELPER FUNCTIONS (defined once)
# ===================================================================================
def _gold_get_prop(tbl, key):
    """Get a table property value as int, or 0 if not set."""
    try:
        props = spark.sql(f"SHOW TBLPROPERTIES {tbl} ('{key}')").collect()
        val = props[0][1] if props else None
        return int(val) if val and val != '(not specified)' and 'does not have property' not in str(val) else 0
    except Exception:
        return 0

def _gold_get_ver(tbl):
    """Get latest version of a Delta table."""
    try:
        return spark.sql(f"SELECT MAX(version) FROM (DESCRIBE HISTORY {tbl})").collect()[0][0] or 0
    except Exception:
        return 0

# ===================================================================================
# CONFIGURATION
# ===================================================================================
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").lower()
tenant_id   = tenant_id_from_customer(customer_id)

load_type = "incremental"  # change to "historical" for full backfill

SILVER_CATALOG     = "hgi"
SILVER_SCHEMA      = "silver"
GOLD_CATALOG       = "hgi"
GOLD_SCHEMA        = "gold"
ENRICHMENT_CATALOG = "hgi"
ENRICHMENT_SCHEMA  = "enrichment"
STALE_DAYS         = 180

FREE_EMAIL_DOMAINS = [
    'gmail.com', 'yahoo.com', 'hotmail.com',
    'aol.com', 'outlook.com', 'icloud.com'
]

spark.conf.set("spark.sql.shuffle.partitions", 4)

print("=" * 70)
print("03_Load_Silver_to_Gold_Augment (OPTIMIZED)")
print(f"  Customer : {customer_id}")
print(f"  Tenant   : {tenant_id}")
print(f"  Load Type: {load_type}")
print("=" * 70)

# ===================================================================================
# CDF PRE-FILTER: Read versions ONCE (used for both CDF check and version tracking)
# ===================================================================================
gold_contacts_all_ver = _gold_get_ver(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all")
gold_accounts_all_ver = _gold_get_ver(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.accounts_all")
gold_tbl_ref          = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations"

# Get all needed properties in one logical section (reused throughout)
last_gold_contacts_cdf_ver = _gold_get_prop(gold_tbl_ref, 'last_contacts_all_cdf_version')
last_gold_accounts_cdf_ver = _gold_get_prop(gold_tbl_ref, 'last_accounts_all_cdf_version')
last_gold_contacts_ver     = _gold_get_prop(gold_tbl_ref, 'last_contacts_all_version')
last_gold_accounts_ver     = _gold_get_prop(gold_tbl_ref, 'last_accounts_all_version')

contacts_cdf_changed = (gold_contacts_all_ver > last_gold_contacts_cdf_ver)
accounts_cdf_changed = (gold_accounts_all_ver > last_gold_accounts_cdf_ver)

print(f"\n  CDF pre-filter:")
print(f"    contacts_all: ver={gold_contacts_all_ver}, last_cdf={last_gold_contacts_cdf_ver}, changed={contacts_cdf_changed}")
print(f"    accounts_all: ver={gold_accounts_all_ver}, last_cdf={last_gold_accounts_cdf_ver}, changed={accounts_cdf_changed}")

# Read CDF for changed EMAILS from contacts_all
cdf_emails_df = None
if contacts_cdf_changed and last_gold_contacts_cdf_ver > 0:
    start_ver = last_gold_contacts_cdf_ver + 1
    try:
        cdf_contacts = (spark.read.format("delta")
            .option("readChangeFeed", "true")
            .option("startingVersion", start_ver)
            .table(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all")
            .filter("_change_type IN ('insert', 'update_postimage')")
            .filter("email IS NOT NULL")
        )
        cdf_emails_df = cdf_contacts.select(F.lower(F.col("email")).alias("email")).distinct()
        cdf_emails_df.createOrReplaceTempView("gold_cdf_emails")
    except Exception as e:
        print(f"    CDF read for contacts_all failed: {e}")
        cdf_emails_df = None
else:
    if not contacts_cdf_changed:
        print(f"    contacts_all unchanged — no email CDF needed")
    else:
        print(f"    First CDF run for contacts — full scan")

# Read CDF for changed DOMAINS from accounts_all
cdf_domains_df = None
if accounts_cdf_changed and last_gold_accounts_cdf_ver > 0:
    start_ver = last_gold_accounts_cdf_ver + 1
    try:
        cdf_accounts = (spark.read.format("delta")
            .option("readChangeFeed", "true")
            .option("startingVersion", start_ver)
            .table(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.accounts_all")
            .filter("_change_type IN ('insert', 'update_postimage')")
            .filter("domain IS NOT NULL")
        )
        cdf_domains_df = cdf_accounts.select(F.lower(F.col("domain")).alias("domain")).distinct()
        cdf_domains_df.createOrReplaceTempView("gold_cdf_domains")
    except Exception as e:
        print(f"    CDF read for accounts_all failed: {e}")
        cdf_domains_df = None
else:
    if not accounts_cdf_changed:
        print(f"    accounts_all unchanged — no domain CDF needed")
    else:
        print(f"    First CDF run for accounts — full scan")

gold_use_cdf_emails  = (cdf_emails_df is not None)
gold_use_cdf_domains = (cdf_domains_df is not None)
print(f"\n  CDF flags: use_cdf_emails={gold_use_cdf_emails}, use_cdf_domains={gold_use_cdf_domains}")

# ===================================================================================
# AUDIT LOGGER INIT
# ===================================================================================
initialize_audit_tables()
print("✅ AuditLogger loaded")

# ===================================================================================
# GOLD TABLE INITIALIZATION — OPT: Guard with tableExists() to skip DDL if exists
# ===================================================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_CATALOG}.{GOLD_SCHEMA}")

def initialize_gold_tables_optimized():
    """Only run CREATE TABLE IF NOT EXISTS when tables don't exist."""
    
    # Check existence with cheap catalog API (1 call each vs 3 DDL roundtrips)
    persons_exists = spark.catalog.tableExists(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.persons")
    companies_exists = spark.catalog.tableExists(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.companies")
    contacts_comp_exists = spark.catalog.tableExists(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations")
    
    if not persons_exists:
        spark.sql(f"""
            CREATE TABLE {GOLD_CATALOG}.{GOLD_SCHEMA}.persons (
                mk_email STRING COMMENT 'Lookup key lowercase email',
                mk_domain STRING, mk_status STRING, mk_timestamp TIMESTAMP,
                name__fullname STRING, name__givenname STRING, name__familyname STRING,
                employment__name STRING, employment__domain STRING, employment__role STRING,
                employment__seniority STRING, employment__title STRING,
                geo__city STRING, geo__country STRING, geo__state STRING,
                is_student BOOLEAN, is_spam BOOLEAN,
                linkedin__handle STRING, twitter__handle STRING, updated_at TIMESTAMP
            ) USING DELTA CLUSTER BY (mk_email)
            TBLPROPERTIES (
                'delta.enableDeletionVectors' = 'true',
                'delta.autoOptimize.optimizeWrite' = 'true',
                'delta.autoOptimize.autoCompact' = 'true',
                'delta.enableChangeDataFeed' = 'true'
            )
        """)
        print("  Created hgi.gold.persons")
    
    if not companies_exists:
        spark.sql(f"""
            CREATE TABLE {GOLD_CATALOG}.{GOLD_SCHEMA}.companies (
                mk_domain STRING COMMENT 'Lookup key lowercase domain',
                mk_status STRING, mk_timestamp TIMESTAMP, name STRING, description STRING,
                category__industry STRING, category__sector STRING,
                category__industrygroup STRING, category__subindustry STRING,
                metrics__employees LONG, metrics__employeesrange STRING,
                metrics__raised LONG, metrics__marketcap STRING,
                geo__city STRING, geo__country STRING, geo__state STRING,
                tech STRING, tags STRING, founded_year LONG, emailprovider BOOLEAN,
                site_url STRING, alexa__globalrank LONG, is_personal BOOLEAN, updated_at TIMESTAMP
            ) USING DELTA CLUSTER BY (mk_domain)
            TBLPROPERTIES (
                'delta.enableDeletionVectors' = 'true',
                'delta.autoOptimize.optimizeWrite' = 'true',
                'delta.autoOptimize.autoCompact' = 'true',
                'delta.enableChangeDataFeed' = 'true'
            )
        """)
        print("  Created hgi.gold.companies")
    
    if not contacts_comp_exists:
        spark.sql(f"""
            CREATE TABLE {GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations (
                contact_id STRING, tenant LONG, email STRING, domain STRING,
                person__mk_email STRING, person__mk_domain STRING, person__mk_status STRING,
                person__mk_timestamp TIMESTAMP, person__name__fullname STRING,
                person__name__givenname STRING, person__name__familyname STRING,
                person__employment__name STRING, person__employment__domain STRING,
                person__employment__role STRING, person__employment__seniority STRING,
                person__employment__title STRING, person__geo__city STRING,
                person__geo__country STRING, person__geo__state STRING,
                person__is_student BOOLEAN, person__is_spam BOOLEAN,
                person__linkedin__handle STRING, person__twitter__handle STRING,
                company__mk_domain STRING, company__mk_status STRING, company__mk_timestamp TIMESTAMP,
                company__name STRING, company__description STRING,
                company__category__industry STRING, company__category__sector STRING,
                company__category__industrygroup STRING, company__category__subindustry STRING,
                company__metrics__employees LONG, company__metrics__employeesrange STRING,
                company__metrics__raised LONG, company__metrics__marketcap STRING,
                company__geo__city STRING, company__geo__country STRING, company__geo__state STRING,
                company__tech STRING, company__tags STRING, company__founded_year LONG,
                company__emailprovider BOOLEAN, company__site_url STRING,
                company__alexa__globalrank LONG, company__is_personal BOOLEAN, updated_at TIMESTAMP
            ) USING DELTA CLUSTER BY (contact_id)
            TBLPROPERTIES (
                'delta.enableDeletionVectors' = 'true',
                'delta.autoOptimize.optimizeWrite' = 'true',
                'delta.autoOptimize.autoCompact' = 'true',
                'delta.enableChangeDataFeed' = 'true'
            )
        """)
        print("  Created hgi.gold.contacts_computations")
    
    if persons_exists and companies_exists and contacts_comp_exists:
        print("  All Gold tables exist — skipped DDL")

initialize_gold_tables_optimized()

# ===================================================================================
# WATERMARK HELPERS
# ===================================================================================
def get_watermark(src_sys, obj_name):
    try:
        result = spark.sql(f"""
            SELECT last_processed_timestamp
            FROM ingestion_metadata.watermarks
            WHERE source_system = '{src_sys}' AND object_name = '{obj_name}'
        """).collect()
        if result:
            return result[0]["last_processed_timestamp"]
    except Exception:
        pass
    return None

def update_watermark(src_sys, obj_name):
    spark.sql(f"""
        MERGE INTO ingestion_metadata.watermarks AS target
        USING (SELECT '{src_sys}' AS source_system, '{obj_name}' AS object_name,
                      current_timestamp() AS last_processed_timestamp) AS source
        ON target.source_system = source.source_system
        AND target.object_name  = source.object_name
        WHEN MATCHED THEN UPDATE SET
            target.last_processed_timestamp = source.last_processed_timestamp
        WHEN NOT MATCHED THEN INSERT
            (source_system, object_name, last_processed_timestamp)
            VALUES (source.source_system, source.object_name, source.last_processed_timestamp)
    """)
    print(f"  Watermark updated: source_system='{src_sys}', object_name='{obj_name}'")

# ===================================================================================
# MAIN PIPELINE
# ===================================================================================
start_time               = time.time()
status                   = "success"
error_type               = None
error_reason             = None
email_count              = 0
domain_count             = 0
persons_enriched_count   = 0
companies_enriched_count = 0
compute_count            = 0
total_rows               = 0

try:
    # ===================================================================================
    # UPSTREAM CHANGE DETECTION — reuse versions from CDF pre-filter (no duplicate calls)
    # ===================================================================================
    contacts_all_ver = gold_contacts_all_ver  # reuse from CDF section
    accounts_all_ver = gold_accounts_all_ver  # reuse from CDF section
    gold_tbl = gold_tbl_ref                   # reuse from CDF section

    contacts_changed = (contacts_all_ver > last_gold_contacts_ver)
    accounts_changed = (accounts_all_ver > last_gold_accounts_ver)
    upstream_changed = contacts_changed or accounts_changed

    print(f"\n  Upstream version check:")
    print(f"    contacts_all: current={contacts_all_ver}, last_gold={last_gold_contacts_ver}, changed={contacts_changed}")
    print(f"    accounts_all: current={accounts_all_ver}, last_gold={last_gold_accounts_ver}, changed={accounts_changed}")

    if load_type == "historical":
        upstream_changed = True
        print(f"    load_type=historical — forcing full processing")

    if not upstream_changed:
        print(f"\n  ✅ No upstream changes — skipping Gold Augment processing")
        email_count = domain_count = persons_enriched_count = companies_enriched_count = compute_count = 0
    else:
        print(f"\n  Changes detected — running Gold Augment pipeline...")

        # -----------------------------------------------------------------------
        # STEP 0A — EMAIL CANDIDATES (OPTIMIZED: no events table JOIN)
        # -----------------------------------------------------------------------
        print("\n" + "="*70)
        print("STEP 0A — EMAIL CANDIDATES (OPTIMIZED)")
        print("="*70)
        
        if load_type == "historical":
            df_email_candidates = spark.sql(f"""
                SELECT DISTINCT lower(email) AS email
                FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all
                WHERE email IS NOT NULL AND tenant = {tenant_id}
            """)
        elif gold_use_cdf_emails:
            # CDF path: only check newly changed emails against persons for staleness
            df_email_candidates = spark.sql(f"""
                SELECT DISTINCT e.email
                FROM gold_cdf_emails e
                LEFT JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.persons p ON e.email = lower(p.mk_email)
                WHERE p.mk_email IS NULL
                   OR p.mk_status IN ('error', 'not_found')
                   OR (p.mk_timestamp IS NOT NULL AND p.mk_timestamp < date_add(current_date(), -{STALE_DAYS}))
            """)
        else:
            # Incremental without CDF: simple contacts_all LEFT JOIN persons
            # PERF: removed 2.15B-row events table JOINs — was ~90s overhead
            df_email_candidates = spark.sql(f"""
                SELECT DISTINCT lower(ca.email) AS email
                FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all ca
                LEFT JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.persons p ON lower(ca.email) = lower(p.mk_email)
                WHERE ca.email IS NOT NULL AND ca.tenant = {tenant_id}
                  AND (p.mk_email IS NULL
                       OR p.mk_status IN ('error', 'not_found')
                       OR (p.mk_timestamp IS NOT NULL AND p.mk_timestamp < date_add(current_date(), -{STALE_DAYS})))
            """)

        # OPT: Use head(1) instead of count() — avoids full action
        email_sample = df_email_candidates.head(1)
        has_email_candidates = len(email_sample) > 0
        if has_email_candidates:
            email_count = df_email_candidates.count()  # only count if we need to proceed
        else:
            email_count = 0
        print(f"  Email candidates: {email_count}")

        # STEP 0B — DOMAIN CANDIDATES (OPTIMIZED)
        # -----------------------------------------------------------------------
        print("\n" + "="*70)
        print("STEP 0B — DOMAIN CANDIDATES (OPTIMIZED)")
        print("="*70)

        if load_type == "historical":
            df_domain_candidates = spark.sql(f"""
                SELECT DISTINCT lower(domain) AS domain
                FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.accounts_all
                WHERE domain IS NOT NULL AND tenant = {tenant_id}
            """)
        elif gold_use_cdf_domains:
            # CDF path: only check newly changed domains against companies
            df_domain_candidates = spark.sql(f"""
                SELECT DISTINCT d.domain
                FROM gold_cdf_domains d
                LEFT JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.companies c ON d.domain = lower(c.mk_domain)
                WHERE c.mk_domain IS NULL
                   OR c.mk_status IN ('error', 'not_found')
                   OR (c.mk_timestamp IS NOT NULL AND c.mk_timestamp < date_add(current_date(), -{STALE_DAYS}))
            """)
        else:
            # Incremental without CDF: simple accounts_all LEFT JOIN companies
            df_domain_candidates = spark.sql(f"""
                SELECT DISTINCT lower(a.domain) AS domain
                FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.accounts_all a
                LEFT JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.companies c ON lower(a.domain) = lower(c.mk_domain)
                WHERE a.domain IS NOT NULL AND a.tenant = {tenant_id}
                  AND (c.mk_domain IS NULL
                       OR c.mk_status IN ('error', 'not_found')
                       OR (c.mk_timestamp IS NOT NULL AND c.mk_timestamp < date_add(current_date(), -{STALE_DAYS})))
            """)

        # OPT: Use head(1) instead of count()
        domain_sample = df_domain_candidates.head(1)
        has_domain_candidates = len(domain_sample) > 0
        if has_domain_candidates:
            domain_count = df_domain_candidates.count()
        else:
            domain_count = 0
        print(f"  Domain candidates: {domain_count}")

        # STEP 1 — EMAIL ENRICHMENT
        # -----------------------------------------------------------------------
        print("\n" + "="*70)
        print("STEP 1 — EMAIL ENRICHMENT")
        print("="*70)

        if email_count > 0:
            df_person_profiles = spark.table(f"{ENRICHMENT_CATALOG}.{ENRICHMENT_SCHEMA}.person_profiles")
            df_persons_enriched = broadcast(df_email_candidates).alias("cand").join(
                df_person_profiles.alias("p"),
                lower(col("cand.email")) == lower(col("p.mk_email")), "left"
            ).select(
                coalesce(lower(col("p.mk_email")), lower(col("cand.email"))).alias("mk_email"),
                when(col("p.mk_domain").isNotNull(), col("p.mk_domain"))
                    .otherwise(when(col("cand.email").contains("@"),
                                    split(lower(col("cand.email")), "@").getItem(1))
                               .otherwise(lit(None))).alias("mk_domain"),
                when(col("p.mk_status").isNotNull(), col("p.mk_status")).otherwise(lit("not_found")).alias("mk_status"),
                when(col("p.mk_timestamp").isNotNull(), col("p.mk_timestamp")).otherwise(current_timestamp()).alias("mk_timestamp"),
                col("p.name__fullname"), col("p.name__givenname"), col("p.name__familyname"),
                col("p.employment__name"), col("p.employment__domain"), col("p.employment__role"),
                col("p.employment__seniority"), col("p.employment__title"),
                col("p.geo__city"), col("p.geo__country"), col("p.geo__state"),
                col("p.is_student"), col("p.is_spam"),
                col("p.linkedin__handle"), col("p.twitter__handle"),
                current_timestamp().alias("updated_at")
            )

            df_persons_deduped = df_persons_enriched.dropDuplicates(["mk_email"])
            DeltaTable.forName(spark, f"{GOLD_CATALOG}.{GOLD_SCHEMA}.persons").alias("target").merge(
                df_persons_deduped.alias("source"),
                "lower(target.mk_email) = lower(source.mk_email)"
            ).whenMatchedUpdate(set={
                "mk_domain":"source.mk_domain","mk_status":"source.mk_status",
                "mk_timestamp":"source.mk_timestamp","name__fullname":"source.name__fullname",
                "name__givenname":"source.name__givenname","name__familyname":"source.name__familyname",
                "employment__name":"source.employment__name","employment__domain":"source.employment__domain",
                "employment__role":"source.employment__role","employment__seniority":"source.employment__seniority",
                "employment__title":"source.employment__title","geo__city":"source.geo__city",
                "geo__country":"source.geo__country","geo__state":"source.geo__state",
                "is_student":"source.is_student","is_spam":"source.is_spam",
                "linkedin__handle":"source.linkedin__handle","twitter__handle":"source.twitter__handle",
                "updated_at":"source.updated_at"
            }).whenNotMatchedInsertAll().execute()
            update_watermark("augment", "persons")
            print(f"  Persons enriched: {email_count}")
        else:
            print(f"  No email candidates — skipping")

        # -----------------------------------------------------------------------
        # STEP 2 — DOMAIN ENRICHMENT
        # -----------------------------------------------------------------------
        print("\n" + "="*70)
        print("STEP 2 — DOMAIN ENRICHMENT")
        print("="*70)

        if domain_count > 0:
            df_company_profiles = spark.table(f"{ENRICHMENT_CATALOG}.{ENRICHMENT_SCHEMA}.company_profiles")
            df_companies_enriched = broadcast(df_domain_candidates).alias("cand").join(
                df_company_profiles.alias("c"),
                lower(col("cand.domain")) == lower(col("c.mk_domain")), "left"
            ).select(
                coalesce(lower(col("c.mk_domain")), lower(col("cand.domain"))).alias("mk_domain"),
                when(col("c.mk_status").isNotNull(), col("c.mk_status")).otherwise(lit("not_found")).alias("mk_status"),
                current_timestamp().alias("mk_timestamp"),
                col("c.name"), col("c.description"),
                col("c.category__industry"), col("c.category__sector"),
                col("c.category__industrygroup"), col("c.category__subindustry"),
                col("c.metrics__employees"), col("c.metrics__employeesrange"),
                col("c.metrics__raised"), col("c.metrics__marketcap"),
                col("c.geo__city"), col("c.geo__country"), col("c.geo__state"),
                col("c.tech"), col("c.tags"), col("c.founded_year"),
                col("c.emailprovider"), col("c.site_url"), col("c.alexa__globalrank"),
                when(lower(col("cand.domain")).isin(FREE_EMAIL_DOMAINS), lit(True))
                    .otherwise(lit(False)).alias("is_personal"),
                current_timestamp().alias("updated_at")
            )

            DeltaTable.forName(spark, f"{GOLD_CATALOG}.{GOLD_SCHEMA}.companies").alias("target").merge(
                df_companies_enriched.alias("source"),
                "lower(target.mk_domain) = lower(source.mk_domain)"
            ).whenMatchedUpdate(set={
                "mk_status":"source.mk_status","mk_timestamp":"source.mk_timestamp",
                "name":"source.name","description":"source.description",
                "category__industry":"source.category__industry","category__sector":"source.category__sector",
                "category__industrygroup":"source.category__industrygroup",
                "category__subindustry":"source.category__subindustry",
                "metrics__employees":"source.metrics__employees",
                "metrics__employeesrange":"source.metrics__employeesrange",
                "metrics__raised":"source.metrics__raised","metrics__marketcap":"source.metrics__marketcap",
                "geo__city":"source.geo__city","geo__country":"source.geo__country",
                "geo__state":"source.geo__state","tech":"source.tech","tags":"source.tags",
                "founded_year":"source.founded_year","emailprovider":"source.emailprovider",
                "site_url":"source.site_url","alexa__globalrank":"source.alexa__globalrank",
                "is_personal":"source.is_personal","updated_at":"source.updated_at"
            }).whenNotMatchedInsertAll().execute()
            update_watermark("augment", "companies")
            print(f"  Companies enriched: {domain_count}")
        else:
            print(f"  No domain candidates — skipping")

        # -----------------------------------------------------------------------
        # STEP 3 — COMPUTE (OPTIMIZED: incremental — only affected contacts)
        # -----------------------------------------------------------------------
        if email_count == 0 and domain_count == 0:
            print(f"\n  No new candidates — skipping contacts_computations rebuild")
            compute_count = spark.sql(f"SELECT COUNT(*) FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations WHERE tenant = {tenant_id}").collect()[0][0]
        else:
            print("\n" + "="*70)
            print("STEP 3 — CONTACTS COMPUTATIONS (INCREMENTAL)")
            print("="*70)

            # Register candidate lists as temp views for incremental filter
            if email_count > 0:
                df_email_candidates.createOrReplaceTempView("gold_enriched_emails")
            if domain_count > 0:
                df_domain_candidates.createOrReplaceTempView("gold_enriched_domains")

            # Build incremental filter — only re-compute contacts affected by enrichment
            inc_filters = []
            if email_count > 0:
                inc_filters.append("lower(ca.email) IN (SELECT email FROM gold_enriched_emails)")
            if domain_count > 0:
                inc_filters.append("lower(ca.domain) IN (SELECT domain FROM gold_enriched_domains)")
            inc_where = " OR ".join(inc_filters) if inc_filters else "1=1"
            
            df_contacts_computations = spark.sql(f"""
                SELECT
                    ca.id AS contact_id, ca.tenant, ca.email, ca.domain,
                    p.mk_email AS person__mk_email, p.mk_domain AS person__mk_domain,
                    p.mk_status AS person__mk_status, p.mk_timestamp AS person__mk_timestamp,
                    p.name__fullname AS person__name__fullname,
                    p.name__givenname AS person__name__givenname,
                    p.name__familyname AS person__name__familyname,
                    p.employment__name AS person__employment__name,
                    p.employment__domain AS person__employment__domain,
                    p.employment__role AS person__employment__role,
                    p.employment__seniority AS person__employment__seniority,
                    p.employment__title AS person__employment__title,
                    p.geo__city AS person__geo__city, p.geo__country AS person__geo__country,
                    p.geo__state AS person__geo__state, p.is_student AS person__is_student,
                    p.is_spam AS person__is_spam, p.linkedin__handle AS person__linkedin__handle,
                    p.twitter__handle AS person__twitter__handle,
                    c.mk_domain AS company__mk_domain, c.mk_status AS company__mk_status,
                    c.mk_timestamp AS company__mk_timestamp, c.name AS company__name,
                    c.description AS company__description,
                    c.category__industry AS company__category__industry,
                    c.category__sector AS company__category__sector,
                    c.category__industrygroup AS company__category__industrygroup,
                    c.category__subindustry AS company__category__subindustry,
                    c.metrics__employees AS company__metrics__employees,
                    c.metrics__employeesrange AS company__metrics__employeesrange,
                    c.metrics__raised AS company__metrics__raised,
                    c.metrics__marketcap AS company__metrics__marketcap,
                    c.geo__city AS company__geo__city, c.geo__country AS company__geo__country,
                    c.geo__state AS company__geo__state, c.tech AS company__tech,
                    c.tags AS company__tags, c.founded_year AS company__founded_year,
                    c.emailprovider AS company__emailprovider, c.site_url AS company__site_url,
                    c.alexa__globalrank AS company__alexa__globalrank,
                    c.is_personal AS company__is_personal,
                    current_timestamp() AS updated_at
                FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all ca
                LEFT JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.persons p ON lower(ca.email) = lower(p.mk_email)
                LEFT JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.companies c ON lower(ca.domain) = lower(c.mk_domain)
                WHERE ca.tenant = {tenant_id} AND ({inc_where})
            """)

            DeltaTable.forName(spark, f"{GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations").alias("target").merge(
                df_contacts_computations.alias("source"),
                "target.contact_id = source.contact_id AND target.tenant = source.tenant"
            ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
            print(f"  Contacts computations merged (incremental: {email_count} emails + {domain_count} domains)")

        # OPT: Batch ALL version markers into a SINGLE ALTER TABLE call (was 4 separate calls)
        spark.sql(f"""ALTER TABLE {gold_tbl} SET TBLPROPERTIES (
            'last_contacts_all_version' = '{contacts_all_ver}',
            'last_accounts_all_version' = '{accounts_all_ver}',
            'last_contacts_all_cdf_version' = '{gold_contacts_all_ver}',
            'last_accounts_all_cdf_version' = '{gold_accounts_all_ver}'
        )""")
        print(f"  Saved ALL version markers in single ALTER TABLE call")

    status = "success"

except Exception as e:
    status       = "failure"
    error_type   = type(e).__name__
    error_reason = str(e)
    print(f"\nERROR: {error_type} — {error_reason}")
    raise

finally:
    # FIX: Compute duration_ms (was commented out, causing NameError)
    end_time    = time.time()
    duration_ms = round((end_time - start_time) * 1000, 2)

    try:
        log_audit(
            job_name         = "03_Load_Silver_to_Gold_Augment",
            customer_id      = customer_id,
            status           = status,
            alert_type       = "SUCCESS" if status == "success" else "FAILURE",
            error_type       = error_type,
            error_reason     = error_reason,
            layer            = "gold",
            rows_processed   = total_rows,
            duration_ms      = int(duration_ms),
            email_on_failure = "pratibha.j.kumari@v4c.ai"
        )
    except Exception as al_e:
        print(f"  WARNING: Audit log failed: {str(al_e)}")

    print(f"""
=======================================================================
  AUGMENT LAYER COMPLETE (OPTIMIZED)
  Customer  : {customer_id}
  Tenant    : {tenant_id}
  Status    : {status.upper()}
  Duration  : {duration_ms/1000:.1f}s
=======================================================================""")